# Xây Dựng Knowledge Base từ TechQA Dataset

Các bước thực hiện bao gồm:
1. Mount (Kết nối) Google Drive.
2. Giải nén file `TechQA.tar.gz` từ Drive.
3. Cài đặt thư viện cần thiết
4. Đọc dữ liệu tài liệu (thường là file `technotes.json`), tiền xử lý, chia nhỏ văn bản (chunking) bằng LangChain.
5. Tạo Embeddings và xây dựng Knowledge Base.
6. Lưu Knowledge Base lên Drive và truy vấn thử.

In [ ]:
# 1. Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Giải nén file TechQA.tar.gz
**Lưu ý:** Bạn cần đảm bảo đã upload file `TechQA.tar.gz` lên Google Drive. Sửa biến `tar_path` bên dưới cho phù hợp với vị trí file trên Drive của bạn.

In [ ]:
import tarfile
import os

tar_path = '/content/drive/MyDrive/TechQA.tar.gz'
extract_path = '/content/TechQA_data'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)

print(f"Đang giải nén {tar_path}...")
try:
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=extract_path)
    print("Giải nén hoàn tất!")
except Exception as e:
    print(f"Lỗi giải nén: {e}\n(Có thể do sai đường dẫn hoặc file bị hỏng)")

print("\nCác thư mục/file sau khi giải nén:")
!ls -la {extract_path}

Đang giải nén /content/drive/MyDrive/TechQA.tar.gz...


/tmp/ipykernel_5922/2194946519.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Giải nén hoàn tất!

Các thư mục/file sau khi giải nén:
total 12
drwxr-xr-x 3 root   root  4096 Aug 14 06:29 .
drwxr-xr-x 1 root   root  4096 Aug 14 06:29 ..
drwxr-sr-x 5 338348 users 4096 Jun 15  2023 TechQA


### 3. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q langchain langchain-huggingface langchain-community sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -q ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 5.4 MB/s eta 0:00:00


### 4. Đọc dữ liệu từ bộ TechQA, tiền xử lý và chia nhỏ văn bản (Chunking)
- Trong bộ TechQA, các tài liệu kỹ thuật được dùng làm Knowledge Base thường được lưu trong file `technotes.json`.
- Mô hình Embedding thường có giới hạn về độ dài văn bản (VD: 512 tokens). Do đó ta cần chia nhỏ các tài liệu dài thành các `chunks` nhỏ hơn.

In [ ]:
import ijson
import glob
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Tìm file
technotes_files = glob.glob(f'{extract_path}/**/full_technote_collection.sections.json', recursive=True)

documents = []
max_docs = 200000  # Lấy 200,000 tài liệu
count = 0

if technotes_files:
    technotes_file = technotes_files[0]
    print(f"Đã tìm thấy file: {technotes_file}")
    print("Đang đọc file JSON theo dạng streaming (tốn rất ít RAM)...")

    # Mở file dưới dạng binary ('rb') bắt buộc đối với ijson
    with open(technotes_file, 'rb') as f:
        try:
            # ijson.kvitems sẽ nhả từng cặp (ID, nội dung) thay vì load toàn bộ file
            # Cách này giả định file của bạn là một Dictionary bọc ngoài
            objects = ijson.kvitems(f, '')
            for doc_id, item in objects:
                if count >= max_docs:
                    break

                # Trích xuất đoạn text
                text = item.get('text') or item.get('document') or item.get('content') or str(item)
                if text:
                    documents.append(Document(page_content=str(text), metadata={"id": doc_id}))
                    count += 1

                    if count % 20000 == 0:
                        print(f" Đã load {count} tài liệu...")

        except Exception as e:
            # Nếu file không phải là Dictionary mà là một List lớn [ {...}, {...} ]
            print(f"Không phải dictionary ({e}). Thử đọc theo dạng danh sách (list)...")
            f.seek(0)
            objects = ijson.items(f, 'item')
            for item in objects:
                if count >= max_docs:
                    break

                text = item.get('text') or item.get('document') or item.get('content') or ''
                doc_id = item.get('id', f'doc_{count}')

                if text:
                    documents.append(Document(page_content=str(text), metadata={"id": doc_id}))
                    count += 1

                    if count % 20000 == 0:
                        print(f" Đã load {count} tài liệu...")

    print(f"\n✅ Hoàn tất đọc dữ liệu! Tổng số: {len(documents)} tài liệu (Giới hạn {max_docs}).")
else:
    print("Không tìm thấy file JSON. Kiểm tra lại đường dẫn!")

# TIẾN HÀNH CHUNKING
if len(documents) > 0:
    print("\nĐang chia nhỏ (chunking) văn bản...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    print(f"✅ Hoàn tất chunking! Tổng cộng: {len(chunks)} chunks.")

Đã tìm thấy file: /content/TechQA_data/TechQA/technote_corpus/full_technote_collection.sections.json
Đang đọc file JSON theo dạng streaming (tốn rất ít RAM)...
 Đã load 20000 tài liệu...
 Đã load 40000 tài liệu...
 Đã load 60000 tài liệu...
 Đã load 80000 tài liệu...
 Đã load 100000 tài liệu...
 Đã load 120000 tài liệu...
 Đã load 140000 tài liệu...
 Đã load 160000 tài liệu...
 Đã load 180000 tài liệu...
 Đã load 200000 tài liệu...

✅ Hoàn tất đọc dữ liệu! Tổng số: 200000 tài liệu (Giới hạn 200000).

Đang chia nhỏ (chunking) văn bản...
✅ Hoàn tất chunking! Tổng cộng: 932222 chunks.


### 5. Khởi tạo Embeddings và xây dựng Vector Database
Biến các đoạn văn bản (chunks) thành vector và lưu trữ vào .

In [ ]:
!pip install -q langchain-qdrant qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 8.9 MB/s eta 0:00:00


In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

# Kiểm tra xem GPU đã được bật chưa
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device.upper()}")
if device == "cpu":
    print("⚠️ CẢNH BÁO: Bạn đang dùng CPU, quá trình embed bằng BGE-M3 sẽ CỰC KỲ LÂU. Hãy bật GPU T4!")

print("Đang tải mô hình BGE-M3 (Sẽ tốn khoảng vài GB tải về)...")
# Khởi tạo mô hình BGE-M3
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': device}
)

# Đường dẫn lưu database Qdrant trên Drive
qdrant_path = "/content/drive/MyDrive/TechQA_Qdrant_BGEM3"
print("Đang tiến hành tạo Vector Database với Qdrant (lưu trực tiếp vào Drive)...")

# Tạo Qdrant local Vector Store 
qdrant = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    path=qdrant_path,
    collection_name="techqa_corpus"
)

print(f"✅ Hoàn tất! Database Qdrant sử dụng BGE-M3 đã được lưu an toàn tại: {qdrant_path}")

Đang sử dụng thiết bị: CUDA
Đang tải mô hình BGE-M3 (Sẽ tốn khoảng vài GB tải về)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Đang tiến hành tạo Vector Database với Qdrant (lưu trực tiếp vào Drive)...


/usr/local/lib/python3.12/dist-packages/langchain_qdrant/qdrant.py:513: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20032 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client.upsert(


### 7. Truy vấn thử (Testing Retriever)

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

# 1. Thiết lập lại thiết bị (ưu tiên GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device.upper()}")

# 2. Khởi tạo lại Embedding Model (BAAI/bge-m3)
print("Đang tải lại mô hình BGE-M3...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': device}
)

# 3. Load lại Qdrant Vector Store từ Google Drive
qdrant_path = "/content/drive/MyDrive/TechQA_Qdrant_BGEM3"
print(f"Đang kết nối tới Database Qdrant tại: {qdrant_path}")

qdrant = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    path=qdrant_path,
    collection_name="techqa_corpus"
)
print("✅ Tải Database hoàn tất!\n")

# 4. Chạy Test Thử Nghiệm Tìm Kiếm
query = "How to configure WebSphere Application Server?"
print(f"🔍 Đang tìm kiếm cho câu hỏi: '{query}'\n")

# Lấy 3 kết quả liên quan nhất (top_k = 3)
results = qdrant.similarity_search(query, k=3)

# In kết quả
if len(results) > 0:
    for i, doc in enumerate(results):
        print(f"--- Kết quả {i+1} ---")
        print(f"📌 Document ID: {doc.metadata.get('id', 'Không có ID')}")
        # In ra 300 ký tự đầu tiên để xem trước nội dung
        print(f"📄 Nội dung: {doc.page_content[:300]}...\n")
else:
    print("Không tìm thấy tài liệu phù hợp.")

Đang sử dụng thiết bị: CUDA
Đang tải lại mô hình BGE-M3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Đang kết nối tới Database Qdrant tại: /content/drive/MyDrive/TechQA_Qdrant_BGEM3


/usr/local/lib/python3.12/dist-packages/langchain_qdrant/qdrant.py:465: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <techqa_corpus> contains 69888 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(


✅ Tải Database hoàn tất!

🔍 Đang tìm kiếm cho câu hỏi: 'How to configure WebSphere Application Server?'

--- Kết quả 1 ---
📌 Document ID: swg21415629
📄 Nội dung: 1) Backup configmgr.ini under <ContentEngine_install_folder>/tools/configure by copying the file and renaming it to configmgr.ini.bak 

2) Edit configmgr.ini and update the vm path to point to the WebSphere Application Server JRE
For example the first 2 lines of configmgr.ini would look like this:
-...

--- Kết quả 2 ---
📌 Document ID: nas8N1012715
📄 Nội dung: Start the Qshell environment. On the i5/OS CL command line, run the STRQSH command and then execute the following commands: 1. cd <app_server_root>/bin 
where app_server_root is the install root of the WebSphere Application Server. [http://publib.boulder.ibm.com/infocenter/wasinfo/v6r1/topic/com.ibm...

--- Kết quả 3 ---
📌 Document ID: swg21651235
📄 Nội dung: Once the WebSphere Application Server interim fixes are installed, you must configure 2 new JVM properties for th